# ML Modeling 

Baseline focus:
- `label_sensitive_binary`
- `label_target_main`

Split settings:
- `random` for in-sample capability
- `state` for cross-state generalization


## 1) Imports


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.svm import LinearSVC

import yaml

from lightgbm import LGBMClassifier

## 2) Paths and Config


In [2]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CFG_PATH = REPO_ROOT / 'config' / 'paths.local.yaml'
if not CFG_PATH.exists():
    CFG_PATH = REPO_ROOT / 'config' / 'paths.template.yaml'
if not CFG_PATH.exists():
    raise FileNotFoundError('Missing config/paths.local.yaml and config/paths.template.yaml')

cfg = yaml.safe_load(CFG_PATH.read_text(encoding='utf-8')) or {}
shared_cfg = cfg.get('shared', {})
repo_cfg = cfg.get('repo', {})

modeling_input = shared_cfg.get('modeling_input')
if not modeling_input:
    raise ValueError('Set shared.modeling_input in YAML')

DATA_PATH = Path(modeling_input)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUNS_BASE_DIR = REPO_ROOT / repo_cfg.get('modeling_runs_dir', 'results/runs/modeling')
RUN_DIR = RUNS_BASE_DIR / f'run_{RUN_ID}'

print('Config:', CFG_PATH)
print('Dataset:', DATA_PATH)
print('Run dir:', RUN_DIR)

Config: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/config/paths.local.yaml
Dataset: /share/ml-data/integrated_all_merged_final.parquet
Run dir: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/results/runs/modeling/run_20260409T051717Z


## 3) Run Settings


In [3]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ROWS = 5_000_000  # None = full dataset. Set int for smoke test.

SPLIT_MODES = ['random', 'state']

STATE_N_SPLITS = 5
STATE_MAX_TRAIN_ROWS = 5_000_000  # Caps train size per fold so large models stay tractable.

DROP_UNKNOWN_TARGET = True  # Drop 'unknown' from label_target_main — labeling artifact, not a real class.

DROP_UNKNOWN_FOR_BINARY = True  # Also drop rows where label_target_main=='unknown' when training binary target.

# Max vendor categories kept as distinct OHE columns; rarer vendors are merged into 'infrequent_category'.
# See Feature Preview (§4b) for recommended threshold if running for the first time.
OHE_MAX_CATEGORIES = 67

_BASE_EXCLUDE = {'label_target_main', 'label_sensitive_binary', 'label_sensitive_subtype', 'add_state'}

## 3b) Model Registry

In [4]:
# Add new models here; they become available to MODELS_TO_RUN without touching run logic.
MODEL_REGISTRY = {
    'logreg': lambda: LogisticRegression(
        solver='saga', max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE,
    ),
    'sgd': lambda: SGDClassifier(
        loss='log_loss', max_iter=1000, tol=1e-4,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=5,
        learning_rate='optimal', class_weight='balanced',
        random_state=RANDOM_STATE, shuffle=True, n_jobs=-1,
    ),
    'rf': lambda: RandomForestClassifier(
        n_estimators=300, max_samples=0.5, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1,
    ),
    'lsvc': lambda: CalibratedClassifierCV(
        LinearSVC(max_iter=2000, class_weight='balanced'),
    ),
    'lgbm': lambda: LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    ),
    'dtree': lambda: DecisionTreeClassifier(
        max_depth=5, 
        class_weight='balanced', 
        random_state=RANDOM_STATE
    )
}

## 3c) Model Selection

In [7]:
# Edit this list to choose models from MODEL_REGISTRY.
MODELS_TO_RUN = ['sgd','dtree']
# e.g. MODELS_TO_RUN = ['logreg', 'sgd']

# Feature representation for vendor_clean.
#   'ohe'        — OneHotEncoder (top OHE_MAX_CATEGORIES vendors, rest → infrequent)
#   'target_enc' — TargetEncoder (vendor → smoothed sensitivity rate; cross-fitting prevents leakage)
#   'both'       — OHE columns + target-encoded column via ColumnTransformer
FEATURE_MODE = 'both'  # 'ohe' | 'target_enc' | 'both'


## 4) Load Dataset + Quick Checkpoints


In [8]:
if DATA_PATH.suffix == '.parquet':
    pl_df = pl.read_parquet(str(DATA_PATH))
else:
    pl_df = pl.read_csv(str(DATA_PATH), separator='\t')

if MAX_ROWS is not None:
    pl_df = pl_df.sample(n=min(MAX_ROWS, pl_df.height), seed=RANDOM_STATE)

pl_df = pl_df.with_columns(pl.col('add_state').cast(pl.String).str.to_uppercase())

print('Shape:', pl_df.shape)
print(pl_df.head(3))

pdf = pl_df.to_pandas()
del pl_df

Shape: (5000000, 5)
shape: (3, 5)
┌───────────┬─────────────────────┬─────────────────────┬────────────────────┬─────────────────────┐
│ add_state ┆ label_sensitive_bin ┆ label_sensitive_sub ┆ label_target_main  ┆ vendor_clean        │
│ ---       ┆ ary                 ┆ type                ┆ ---                ┆ ---                 │
│ str       ┆ ---                 ┆ ---                 ┆ str                ┆ str                 │
│           ┆ i8                  ┆ str                 ┆                    ┆                     │
╞═══════════╪═════════════════════╪═════════════════════╪════════════════════╪═════════════════════╡
│ MARYLAND  ┆ 0                   ┆ non_sensitive       ┆ unknown            ┆ Vantiva             │
│ NEW YORK  ┆ 0                   ┆ non_sensitive       ┆ residential_place  ┆ CHONGQING FUGUI     │
│           ┆                     ┆                     ┆                    ┆ ELECTRONICS CO…     │
│ ALABAMA   ┆ 0                   ┆ non_sensitive       ┆

## 4b) Feature Preview


In [9]:
print(f'All dataset columns ({len(pdf.columns)}): {list(pdf.columns)}')
print(f'Always excluded: {sorted(_BASE_EXCLUDE)}\n')

vc = pdf['vendor_clean'].value_counts()
ohe_width = min(len(vc), OHE_MAX_CATEGORIES)
coverage  = vc.head(OHE_MAX_CATEGORIES).sum() / vc.sum()

target_coverage = 0.95
n_for_target = (vc.cumsum() / vc.sum() <= target_coverage).sum() + 1

print(f'vendor_clean: {len(vc)} unique → {ohe_width} OHE cols')
print(f'  top-{OHE_MAX_CATEGORIES} coverage: {coverage:.1%} of rows')
print(f'  vendors needed for {target_coverage:.0%} coverage: {n_for_target}  (current OHE_MAX_CATEGORIES={OHE_MAX_CATEGORIES})')
print(f'\nModels to run: {MODELS_TO_RUN}')

All dataset columns (5): ['add_state', 'label_sensitive_binary', 'label_sensitive_subtype', 'label_target_main', 'vendor_clean']
Always excluded: ['add_state', 'label_sensitive_binary', 'label_sensitive_subtype', 'label_target_main']

vendor_clean: 1343 unique → 67 OHE cols
  top-67 coverage: 95.0% of rows
  vendors needed for 95% coverage: 67  (current OHE_MAX_CATEGORIES=67)

Models to run: ['sgd', 'dtree']


## 5) Modeling Helpers

### Preprocessing
- **`OneHotEncoder`**: expands `vendor_clean` into one 0/1 column per vendor; rare vendors (beyond `OHE_MAX_CATEGORIES`) bundled into `infrequent_category`; `sparse_output=True` keeps the matrix sparse through to the classifier

### Split strategies
- **`random`**: stratified 80/20 — class ratios preserved; tests in-sample capability
- **`state`**: `StratifiedGroupKFold` — all records from the same state stay together; test folds contain only unseen states; 5 folds = train on 4/5 of states, test on 1/5; tests geographic generalization

### Helpers
- **`build_split_folds`**: returns `[(fold_id, train_idx, test_idx)]` for the chosen split strategy
- **`evaluate_model`**: accuracy, macro/weighted F1, confusion matrix, classification report; adds PR-AUC + ROC-AUC for binary targets
- **`run_for_target`**: runs all split modes × folds × `MODELS_TO_RUN` for a single target column; returns list of result dicts

In [10]:
def build_split_folds(y, split_mode, state_series=None):
    """Return list of (fold_id, train_idx, test_idx) for the given split strategy."""

    if split_mode == 'random':
        train_idx, test_idx = train_test_split(
            y.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
        )
        return [('random_fold_1', pd.Index(train_idx), pd.Index(test_idx))]

    if split_mode == 'state':
        groups = state_series.astype('string').str.upper().str.strip()
        valid  = groups.notna() & (groups != '')
        y_v    = y.loc[valid]
        g_v    = groups.loc[valid]

        n_groups = int(g_v.nunique())
        if n_groups < 3:
            raise ValueError(f'Need >=3 states for grouped CV, got {n_groups}')

        n_splits = min(STATE_N_SPLITS, n_groups)
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        folds = []
        idx = y_v.index
        for i, (tr, te) in enumerate(splitter.split(np.zeros((len(y_v), 1)), y_v, g_v), start=1):
            folds.append((f'state_fold_{i}', pd.Index(idx[tr]), pd.Index(idx[te])))

        print(f'  State split: {n_groups} states/territories → {n_splits} folds')
        return folds

    raise ValueError(f'Unknown split_mode: {split_mode}')

In [11]:
def evaluate_model(pipe, X_test, y_test):
    """Compute classification metrics. For binary targets also computes PR-AUC and ROC-AUC."""
    pred = pipe.predict(X_test)
    out = {
        'accuracy':    float(accuracy_score(y_test, pred)),
        # macro_f1: equal weight per class — fair to minority classes.
        'macro_f1':    float(f1_score(y_test, pred, average='macro',    zero_division=0)),
        # weighted_f1: weighted by class size — dominated by majority class.
        'weighted_f1': float(f1_score(y_test, pred, average='weighted', zero_division=0)),
        'confusion_matrix':      confusion_matrix(y_test, pred),
        'classification_report': classification_report(y_test, pred, output_dict=True, zero_division=0),
    }
    # PR-AUC is more informative than ROC-AUC under class imbalance.
    if len(pd.Series(y_test).dropna().unique()) == 2 and hasattr(pipe, 'predict_proba'):
        proba = pipe.predict_proba(X_test)[:, 1]
        out['pr_auc']  = float(average_precision_score(y_test, proba))
        out['roc_auc'] = float(roc_auc_score(y_test, proba))
    return out


In [12]:
def _build_pipeline(model_name):
    """Return preprocessing + classifier pipeline for the current FEATURE_MODE."""
    ohe = OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        max_categories=OHE_MAX_CATEGORIES,
        sparse_output=True,
    )
    te = TargetEncoder(random_state=RANDOM_STATE)

    if FEATURE_MODE == 'ohe':
        steps = [('ohe', ohe)]
    elif FEATURE_MODE == 'target_enc':
        steps = [('target_enc', te)]
    elif FEATURE_MODE == 'both':
        steps = [('features', ColumnTransformer([
            ('ohe',        ohe, ['vendor_clean']),
            ('target_enc', te,  ['vendor_clean']),
        ], sparse_threshold=0.3))]
    else:
        raise ValueError(f'Unknown FEATURE_MODE: {FEATURE_MODE!r}')

    steps.append(('classifier', MODEL_REGISTRY[model_name]()))
    return Pipeline(steps)


def run_for_target(target_col, df):
    """Run all split modes x folds x MODELS_TO_RUN for one target column."""
    work_df = df[df[target_col].notna()].copy()

    if target_col == 'label_target_main' and DROP_UNKNOWN_TARGET:
        before  = len(work_df)
        work_df = work_df[work_df[target_col] != 'unknown'].copy()
        print(f'  Dropped {before - len(work_df):,} "unknown" rows')

    if target_col == 'label_sensitive_binary' and DROP_UNKNOWN_FOR_BINARY:

        before  = len(work_df)

        work_df = work_df[work_df['label_target_main'] != 'unknown'].copy()

        print(f'  Dropped {before - len(work_df):,} rows where label_target_main=\"unknown\" for binary target')

    y_full       = work_df[target_col]
    state_series = work_df['add_state']

    print(f'\nTarget={target_col}, n={len(work_df):,}, models={MODELS_TO_RUN}')

    results = []
    for split_mode in SPLIT_MODES:
        folds = build_split_folds(y_full, split_mode=split_mode, state_series=state_series)

        for fold_id, train_idx, test_idx in folds:
            if split_mode == 'state' and STATE_MAX_TRAIN_ROWS and len(train_idx) > STATE_MAX_TRAIN_ROWS:
                rng       = np.random.default_rng(RANDOM_STATE)
                train_idx = pd.Index(rng.choice(train_idx, size=STATE_MAX_TRAIN_ROWS, replace=False))

            X_train = work_df.loc[train_idx, ['vendor_clean']]
            X_test  = work_df.loc[test_idx,  ['vendor_clean']]
            y_train = y_full.loc[train_idx]
            y_test  = y_full.loc[test_idx]

            for model_name in MODELS_TO_RUN:
                pipe = _build_pipeline(model_name)
                pipe.fit(X_train, y_train)
                metric = evaluate_model(pipe, X_test, y_test)

                out = {
                    'target': target_col, 'split_mode': split_mode,
                    'fold_id': fold_id,   'model': model_name,
                    'feature_mode': FEATURE_MODE,
                    'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
                    **metric,
                }
                results.append(out)
                print(f'  {target_col} | {split_mode} | {fold_id} | {model_name} | {FEATURE_MODE} → macro_f1={out["macro_f1"]:.4f}')

    return results

## 6a) Run → `label_target_main`

Run this cell independently to train and evaluate on the land-use target only.


In [ ]:
results_main = run_for_target('label_target_main', pdf)
print('Total runs (label_target_main):', len(results_main))


## 6b) Run → `label_sensitive_binary`

Run this cell independently to train and evaluate on the sensitive/non-sensitive binary target only.


In [13]:
results_binary = run_for_target('label_sensitive_binary', pdf)
print('Total runs (label_sensitive_binary):', len(results_binary))


  Dropped 594,977 rows where label_target_main="unknown" for binary target

Target=label_sensitive_binary, n=4,405,023, models=['sgd', 'dtree']
  label_sensitive_binary | random | random_fold_1 | sgd | both → macro_f1=0.6256
  label_sensitive_binary | random | random_fold_1 | dtree | both → macro_f1=0.6208
  State split: 56 states/territories → 5 folds
  label_sensitive_binary | state | state_fold_1 | sgd | both → macro_f1=0.6320
  label_sensitive_binary | state | state_fold_1 | dtree | both → macro_f1=0.6266
  label_sensitive_binary | state | state_fold_2 | sgd | both → macro_f1=0.6205
  label_sensitive_binary | state | state_fold_2 | dtree | both → macro_f1=0.6175
  label_sensitive_binary | state | state_fold_3 | sgd | both → macro_f1=0.6252
  label_sensitive_binary | state | state_fold_3 | dtree | both → macro_f1=0.6210
  label_sensitive_binary | state | state_fold_4 | sgd | both → macro_f1=0.6212
  label_sensitive_binary | state | state_fold_4 | dtree | both → macro_f1=0.6175
  lab

## 7) Summary + Save

In [14]:
# Combine whichever result lists were populated — safe if only one target cell was run.
all_results = [
    *globals().get('results_main',   []),
    *globals().get('results_binary', []),
]

summary = pd.DataFrame([
    {
        'target':      r['target'],
        'split_mode':  r['split_mode'],
        'fold_id':     r.get('fold_id', 'na'),
        'model':       r['model'],
        'feature_mode': r.get('feature_mode', 'ohe'),
        'n_train':     r['n_train'],
        'n_test':      r['n_test'],
        'accuracy':    r['accuracy'],
        'macro_f1':    r['macro_f1'],
        'weighted_f1': r['weighted_f1'],
        'pr_auc':      r.get('pr_auc',  float('nan')),
        'roc_auc':     r.get('roc_auc', float('nan')),
    }
    for r in all_results
]).sort_values(['target', 'split_mode', 'model', 'feature_mode', 'fold_id'])

RUN_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(RUN_DIR / 'summary_metrics.csv', index=False)
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(
    {'run_id': RUN_ID, 'data_path': str(DATA_PATH), 'models_run': MODELS_TO_RUN, 'feature_mode': FEATURE_MODE},
    indent=2,
))
print('Saved to:', RUN_DIR)
display(summary)

Saved to: /share/project_repo/zzhan342/WiFiMLCapstoneS2026/results/runs/modeling/run_20260409T051717Z


,target,split_mode,fold_id,model,feature_mode,n_train,n_test,accuracy,macro_f1,weighted_f1,pr_auc,roc_auc
1,label_sensitive_binary,random,random_fold_1,dtree,both,3524018,881005,0.839607,0.620814,0.873182,0.247385,0.817598
0,label_sensitive_binary,random,random_fold_1,sgd,both,3524018,881005,0.846175,0.625581,0.877387,0.247071,0.816206
3,label_sensitive_binary,state,state_fold_1,dtree,both,3550065,854956,0.843733,0.626634,0.875240,0.250012,0.814136
5,label_sensitive_binary,state,state_fold_2,dtree,both,3565388,839633,0.832352,0.617473,0.866846,0.256149,0.815393
7,label_sensitive_binary,state,state_fold_3,dtree,both,3450182,954839,0.840370,0.620976,0.875654,0.263416,0.831713
9,label_sensitive_binary,state,state_fold_4,dtree,both,3500086,904935,0.844749,0.617479,0.877235,0.229091,0.814306
11,label_sensitive_binary,state,state_fold_5,dtree,both,3554363,850658,0.837372,0.622321,0.870974,0.236135,0.813213
2,label_sensitive_binary,state,state_fold_1,sgd,both,3550065,854956,0.850825,0.631986,0.879791,0.250596,0.812941
4,label_sensitive_binary,state,state_fold_2,sgd,both,3565388,839633,0.836970,0.620485,0.869801,0.256269,0.813907
6,label_sensitive_binary,state,state_fold_3,sgd,both,3450182,954839,0.846247,0.625211,0.879408,0.259998,0.829995


## 8) Detailed Metrics (Optional)

In [15]:
for r in all_results:
    label = f"{r['target']}  ·  {r['split_mode']}  ·  {r['fold_id']}  ·  {r['model']}"
    print(f'\n{"─" * 72}\n{label}\n')

    classes = [k for k in r['classification_report'] if k not in ('accuracy', 'macro avg', 'weighted avg')]
    cm = pd.DataFrame(r['confusion_matrix'], index=classes, columns=classes)
    cm.index.name   = 'actual \\ pred'
    display(cm)

    print()
    display(pd.DataFrame(r['classification_report']).T.round(3))


────────────────────────────────────────────────────────────────────────
label_sensitive_binary  ·  random  ·  random_fold_1  ·  sgd



,0,1
actual \ pred,,
0,710858,115602
1,19919,34626


,precision,recall,f1-score,support
0,0.973,0.860,0.913,826460.000
1,0.230,0.635,0.338,54545.000
accuracy,0.846,0.846,0.846,0.846
macro avg,0.602,0.747,0.626,881005.000
weighted avg,0.927,0.846,0.877,881005.000



────────────────────────────────────────────────────────────────────────
label_sensitive_binary  ·  random  ·  random_fold_1  ·  dtree



,0,1
actual \ pred,,
0,704459,122001
1,19306,35239


...

,0,1
actual \ pred,,
0,681598,114828
1,19549,34683


,precision,recall,f1-score,support
0,0.972,0.856,0.910,796426.000
1,0.232,0.640,0.340,54232.000
accuracy,0.842,0.842,0.842,0.842
macro avg,0.602,0.748,0.625,850658.000
weighted avg,0.925,0.842,0.874,850658.000



────────────────────────────────────────────────────────────────────────
label_sensitive_binary  ·  state  ·  state_fold_5  ·  dtree



,0,1
actual \ pred,,
0,677106,119320
1,19021,35211


,precision,recall,f1-score,support
0,0.973,0.850,0.907,796426.000
1,0.228,0.649,0.337,54232.000
accuracy,0.837,0.837,0.837,0.837
macro avg,0.600,0.750,0.622,850658.000
weighted avg,0.925,0.837,0.871,850658.000
